## Random File Upload with Cloudflare R2 Persistence

A compact end-to-end example that creates a random binary file and stores it in Cloudflare R2 using LAILA.

### What this notebook does
- Create a random binary file in the current directory (`./random_blob.bin`)
- Read file bytes and wrap them as a LAILA entry (`laila.constant`) to get a global ID
- Persist the entry to Cloudflare via `CloudflarePool` (`memorize`)
- Retrieve the entry from Cloudflare (`remember`) and validate integrity
- Remove the stored object (`forget`) for cleanup

### Credentials
This notebook reads Cloudflare credentials from:
`laila.args`


In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import sys
sys.path.append("/home/ubuntu/")

In [3]:
import laila
laila.read_args("/home/ubuntu/laila/vault/dev_secrets.toml")

In [4]:
from laila.pool import CloudflarePool

required = ["CLOUDFLARE_ACCOUNT_ID", "CLOUDFLARE_ACCESS_KEY_ID", "CLOUDFLARE_SECRET_ACCESS_KEY", "CLOUDFLARE_BUCKET_NAME"]
missing = [name for name in required if getattr(laila.args, name, None) is None]
if missing:
    raise RuntimeError(f"Missing laila.args values: {', '.join(missing)}")

cloudflare_pool = CloudflarePool(
    account_id=laila.args.CLOUDFLARE_ACCOUNT_ID,
    access_key_id=laila.args.CLOUDFLARE_ACCESS_KEY_ID,
    secret_access_key=laila.args.CLOUDFLARE_SECRET_ACCESS_KEY,
    bucket_name=laila.args.CLOUDFLARE_BUCKET_NAME,
)

In [5]:
laila.add_pool(cloudflare_pool, pool_nickname="file_backup_cf_pool")

In [6]:
# Create a random binary file in the current directory (.)
from pathlib import Path
import os

file_path = Path("./random_blob.dev.bin")
file_size_bytes = 16 * 1024 * 1024 # 16 MB

with open(file_path, "wb") as f:
    f.write(os.urandom(file_size_bytes))

print(f"Created: {file_path.resolve()} ({file_size_bytes} bytes)")

Created: /home/ubuntu/laila/examples/simple/random_blob.dev.bin (16777216 bytes)


In [7]:
#read the file into a buffer
with open(file_path, "rb") as f:
    buffer = f.read()
#wrap the buffer in a constant to get a global id
wrapped_buffer = laila.constant(data=buffer)
print (wrapped_buffer.global_id)
#store the result in Cloudflare
future_memorize = laila.memorize(wrapped_buffer, pool_nickname="file_backup_cf_pool")
print ("Memorizing to cloudflare...")
future_memorize.wait()
print ("Memorized.")

LAILA:ENTRY:GLOBAL_ID:1cab2b9b-14bb-4c32-8c06-d14ed6668d30
Memorizing to cloudflare...
Memorized.


In [9]:
#remembering the file
future_remember = laila.remember("LAILA:ENTRY:GLOBAL_ID:1cab2b9b-14bb-4c32-8c06-d14ed6668d30", pool_nickname="file_backup_cf_pool")
future_remember.wait()
print ("Recovered.")


Recovered.


In [ ]:
assert future_remember.result.data == wrapped_buffer.data
print ("Data matches.")

Data matches.
